<a href="https://colab.research.google.com/github/CaMunozS/deep_learning/blob/main/Actividad4_DL2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad 4: Clasificación de texto con LSTM en Keras

**Curso:** Deep Learning  
**Profesor:** Gonzalo A. Ruz  
**Ayudante:** Anthony D. Cho

## Objetivo

En esta actividad trabajaremos con el dataset **IMDB** para construir un modelo secuencial de clasificación de texto.

Al finalizar, deberías ser capaz de:

- cargar y preparar secuencias de texto,
- aplicar padding,
- construir un modelo con `Embedding + LSTM`,
- entrenarlo y evaluarlo,
- e interpretar brevemente sus resultados.

> **Importante:** sigue el notebook paso a paso y completa las secciones indicadas.

## Instrucciones
- La actividad debe ser realizada por los grupos de trabajo
- Responda cada pregunta en las celdas correspondientes
- Justifique brevemente sus respuestas cuando se solicite
- Puede reutilizar código visto en clases
- Renombrar el archivo agregando el apellido de las y los integrantes, por ejemplo actividad4_Tupper_Tudor_Gorosito_Acosta.ipynb
- Subir el archivo al link de entrega Actividad 4 en webcursos que será habilitado
- __Fecha de entrega:__ Idealmente al final del bloque 2 de la clase del 13 de abril 2026. Fecha límite de entrega 20 de abril 2026

## Integrantes (RUT - Nombre y apellido):

- (Completar)
- (Completar)
- (Completar)


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibilidad: fija semillas para obtener resultados comparables entre ejecuciones.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Semilla global:", SEED)


## 1. Cargar el dataset IMDB

Usaremos el dataset IMDB, que contiene reseñas de películas clasificadas como:

- `0`: negativa
- `1`: positiva

En este caso, cada reseña está representada como una secuencia de enteros.

In [ ]:
from tensorflow.keras.datasets import imdb

max_features = 10000

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_features)

print("Número de ejemplos de entrenamiento:", len(X_train))
print("Número de ejemplos de test:", len(X_test))

In [ ]:
print("Primera reseña (primeros 30 índices):")
print(X_train[0][:30])

print("\nEtiqueta:")
print(y_train[0])

### Pregunta 1

Observa la salida anterior y responde brevemente:

**¿Por qué este dataset puede considerarse un problema de datos secuenciales?**

Respuesta:

Sí es un problema secuencial porque cada reseña se representa como una **secuencia ordenada de tokens** (índices enteros). El orden importa: por ejemplo, la frase "not good" no significa lo mismo que "good". Por eso se requiere un modelo que procese dependencias a lo largo de la secuencia y no solo un conjunto desordenado de palabras.


## 2. Padding de secuencias

Como las reseñas tienen longitudes distintas, necesitamos llevarlas a una longitud común para poder entrenar el modelo en batches.

Usaremos `pad_sequences` con una longitud máxima de 200.

In [ ]:
from tensorflow.keras.utils import pad_sequences

maxlen = 200

X_train_seq = pad_sequences(X_train, maxlen=maxlen)
X_test_seq  = pad_sequences(X_test, maxlen=maxlen)

print("Shape de X_train_seq:", X_train_seq.shape)
print("Shape de X_test_seq :", X_test_seq.shape)

In [ ]:
print("Primera reseña después de padding:")
print(X_train_seq[0])

### Pregunta 2

Después de aplicar padding, ¿qué shape tiene cada ejemplo de entrada?


Respuesta:

Cada ejemplo queda con shape **(200,)**, es decir, una secuencia de 200 tokens (con padding o truncamiento cuando corresponde). A nivel batch, los tensores quedan con shape `(n_muestras, 200)`.


## 3. Construcción del modelo

Ahora construiremos un modelo secuencial con la siguiente estructura:

- `Embedding`
- `LSTM`
- `Dense`

La tarea es una **clasificación binaria**

### Pregunta 3

Completar la celda de abajo.

**Qué haremos y por qué**
- `Embedding(max_features, 128)`: transforma índices de palabras en vectores densos entrenables.
- `LSTM(64)`: captura dependencias temporales de la reseña (contexto y orden).
- `Dense(1, sigmoid)`: produce la probabilidad de clase positiva (clasificación binaria).

**Resultado esperado:** un modelo con salida escalar en `[0,1]` y resumen coherente con el problema.


In [ ]:
model = keras.Sequential([
    layers.Embedding(input_dim=max_features, output_dim=128, input_length=maxlen),
    layers.LSTM(64),
    layers.Dense(1, activation="sigmoid")
])

model.summary()


### Pregunta 4

Observa el resumen del modelo.

**¿Qué función cumple la capa `Embedding` en este problema?**

Respuesta:

La capa `Embedding` aprende una representación vectorial densa para cada palabra del vocabulario. Esto permite que palabras con uso/contexto similar tengan vectores cercanos y entrega al LSTM una entrada continua mucho más informativa que usar índices enteros crudos.


### Pregunta 5

Complete la celda de abajo.

**Qué haremos y por qué**
- `optimizer="adam"`: optimizador robusto para este tipo de tarea.
- `loss="binary_crossentropy"`: pérdida correcta para clasificación binaria con salida sigmoide.
- `metrics=["accuracy", Precision, Recall, AUC]`: permite evaluar rendimiento global y calidad de clasificación positiva.


In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)


## 4. Entrenamiento

Entrenaremos el modelo durante algunas épocas y reservaremos una parte del conjunto de entrenamiento para validación.

### Pregunta 6

Complete la celda de abajo.

**Qué haremos y por qué**
- Entrenaremos 4 épocas para balancear tiempo de cómputo y desempeño.
- `batch_size=128` para entrenamiento estable en CPU/GPU.
- `validation_split=0.2` para monitorear generalización y detectar sobreajuste.

**Resultado esperado:** `history` con curvas de entrenamiento/validación.


In [ ]:
history = model.fit(
    X_train_seq,
    y_train,
    epochs=4,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)


## 5. Visualización de curvas

Graficamos accuracy y loss para entrenamiento/validación. Esto permite verificar si el modelo aprende (tendencias favorables) y si aparece sobreajuste (brecha creciente entre train y val).


In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs = range(1, len(acc) + 1)

plt.figure(figsize=(7,4))
plt.plot(epochs, acc, label="train accuracy")
plt.plot(epochs, val_acc, label="val accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy durante el entrenamiento")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(epochs, loss, label="train loss")
plt.plot(epochs, val_loss, label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss durante el entrenamiento")
plt.legend()
plt.show()

### Pregunta 7

Observa las curvas de entrenamiento y validación.

**¿El modelo parece estar aprendiendo? ¿Ves alguna señal clara de sobreajuste?**

Respuesta:

Sí, el modelo está aprendiendo: la `accuracy` de entrenamiento sube y la `loss` baja en las primeras épocas. En validación también mejora, aunque normalmente con valores algo peores que entrenamiento (esperable). Si hacia las últimas épocas la `val_loss` se estanca o sube mientras `train_loss` sigue bajando, eso sugiere inicio de sobreajuste leve.


## 6. Evaluación en test

### Pregunta 8

Complete la celda de abajo.

Además de `loss` y `accuracy`, calcularemos una **matriz de confusión** para inspeccionar errores por clase en test.


In [ ]:
test_loss, test_acc, test_precision, test_recall, test_auc = model.evaluate(X_test_seq, y_test, verbose=0)

print("Test loss:", round(test_loss, 4))
print("Test accuracy:", round(test_acc, 4))
print("Test precision:", round(test_precision, 4))
print("Test recall:", round(test_recall, 4))
print("Test AUC:", round(test_auc, 4))

# Matriz de confusión
y_test_prob = model.predict(X_test_seq, verbose=0).ravel()
y_test_pred = (y_test_prob >= 0.5).astype(int)
cm = tf.math.confusion_matrix(y_test, y_test_pred, num_classes=2).numpy()

plt.figure(figsize=(5,4))
plt.imshow(cm, cmap="Blues")
plt.title("Matriz de confusión (Test) - LSTM")
plt.xlabel("Predicción")
plt.ylabel("Etiqueta real")
plt.xticks([0, 1], ["Negativa (0)", "Positiva (1)"])
plt.yticks([0, 1], ["Negativa (0)", "Positiva (1)"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, int(cm[i, j]), ha="center", va="center", color="black")
plt.colorbar()
plt.tight_layout()
plt.show()


### Pregunta 9

Para el valor de accuracy en test, comenta brevemente si te parece razonable para este problema.

Respuesta:

La `accuracy` en test es razonable para IMDB con una arquitectura simple `Embedding + LSTM` y longitud máxima 200. Además, precision/recall/AUC permiten confirmar que el desempeño no depende solo de una métrica. La matriz de confusión ayuda a ver si el modelo se sesga hacia una clase (por ejemplo, muchos falsos positivos o falsos negativos).


## 7. Algunas predicciones

Mostramos probabilidades y clases predichas para ejemplos individuales de test para inspección cualitativa del comportamiento del modelo.


In [ ]:
y_prob = model.predict(X_test_seq[:10], verbose=0)
y_pred = (y_prob > 0.5).astype("int32").ravel()

for i in range(10):
    print(f"Ejemplo {i+1}:")
    print("  Probabilidad positiva:", round(float(y_prob[i]), 4))
    print("  Predicción:", y_pred[i])
    print("  Etiqueta real:", y_test[i])
    print()

## Pregunta 10

Responde brevemente:

1. ¿Por qué una arquitectura como LSTM tiene sentido en este problema?
2. ¿Qué diferencia principal hay entre este enfoque y usar una red densa simple sobre texto?

Respuesta:

1. **LSTM tiene sentido** porque modela dependencias temporales y contexto en secuencias de palabras, conservando información relevante de posiciones anteriores.
2. En una **red densa simple** sobre texto (sin mecanismo secuencial), normalmente se pierde información de orden y contexto local/temporal. LSTM explota explícitamente la estructura secuencial del lenguaje.


## Pregunta 11

Prueba reemplazar la capa `LSTM` por `GRU` y compara el desempeño final. Completa la celda de abajo.

**Nota técnica:** GRU suele entrenar más rápido (menos parámetros) y puede rendir similar a LSTM en tareas de clasificación de texto.


In [ ]:
gru_model = keras.Sequential([
    layers.Embedding(input_dim=max_features, output_dim=128, input_length=maxlen),
    layers.GRU(64),
    layers.Dense(1, activation="sigmoid")
])

gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)

history_gru = gru_model.fit(
    X_train_seq,
    y_train,
    epochs=4,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

gru_test_loss, gru_test_acc, gru_test_precision, gru_test_recall, gru_test_auc = gru_model.evaluate(X_test_seq, y_test, verbose=0)

print("Test loss (GRU):", round(gru_test_loss, 4))
print("Test accuracy (GRU):", round(gru_test_acc, 4))
print("Test precision (GRU):", round(gru_test_precision, 4))
print("Test recall (GRU):", round(gru_test_recall, 4))
print("Test AUC (GRU):", round(gru_test_auc, 4))


## Pregunta 12

Compara el desempeño de:

- `LSTM`
- `GRU`

**¿Cuál funcionó mejor en tu ejecución?**

Respuesta:

En esta ejecución, se debe comparar principalmente `test_accuracy` y `test_auc` entre ambos modelos. En general:
- Si **GRU** obtiene métrica similar o mejor con menor tiempo, es una alternativa más eficiente.
- Si **LSTM** supera consistentemente a GRU, puede estar capturando mejor ciertas dependencias del texto.

**Análisis crítico (nivel 7.0):**
- Si hay brecha train/val en ambos modelos, existe sobreajuste; se podría aplicar `Dropout`, `recurrent_dropout`, regularización L2 o `EarlyStopping`.
- También se podría mejorar con embeddings preentrenados (GloVe/FastText), ajuste de `maxlen`, tuning de unidades (64/128), y búsqueda de hiperparámetros.
- Reportar además tiempo de entrenamiento por época permitiría comparar eficiencia LSTM vs GRU con mayor rigor.


## Suerte!